# Numerix x Chronos-2: Headline Demo

**Goal:** show that augmenting a parametric MC engine with Chronos-2 *changes the answer in regimes where the parametric engine is structurally weak*, while leaving the answer essentially unchanged in calm regimes.

Beats:
1. Pull the joint FX+rates panel and the XCCY book.
2. Pick an as-of date in a known stress window.
3. Run the parametric engine.
4. Run Chronos-2 (with covariates).
5. Blend by `w_stress` from the regime detector.
6. Compute EE / PFE@97.5 / ES@97.5 for baseline vs augmented.
7. Render the money-shot overlay + diagnostics.

In [ ]:
%load_ext autoreload
%autoreload 2

import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronos2_demo import config
from chronos2_demo.augmentation import blender, regime_detector
from chronos2_demo.data import covariates as cov_loader
from chronos2_demo.diagnostics import coverage, distribution, tail_dependence
from chronos2_demo.models import baseline_mc, chronos2_client
from chronos2_demo.pricing import book, exposure
from chronos2_demo.plotting import path_samples, pfe_overlay, regime_ribbon

## Beat 1 — Load the curated panel and the book

In [ ]:
panel = pd.read_parquet(config.DATA_CURATED / 'panel.parquet')
covs = pd.read_parquet(config.DATA_CURATED / 'covariates.parquet')
swap_book = book.load_book(book.DEFAULT_BOOK_PATH)
print(f'panel: {panel.shape}, covariates: {covs.shape}, swaps: {len(swap_book)}')

## Beat 2 — Pick an as-of date and look at the regime snapshot

In [ ]:
AS_OF = config.DEFAULT_AS_OF  # 2022-06-30
HORIZON = config.DEFAULT_HORIZON_DAYS
N_PATHS = config.DEFAULT_N_PATHS

snap = regime_detector.current_w_stress(panel, as_of=AS_OF, window=20)
print(f'w_stress at {AS_OF}: {snap.w_stress:.3f}')
print(snap.features)

## Beat 3 — Run the parametric engine

In [ ]:
par_paths = baseline_mc.simulate_baseline(
    panel, as_of=AS_OF, horizon=HORIZON, n_paths=N_PATHS,
    n_components=5, seed=0,
)
print(par_paths.source, par_paths.paths.shape)

## Beat 4 — Run Chronos-2 (real-time SageMaker endpoint)

In [ ]:
runtime = boto3.client('sagemaker-runtime', region_name='us-east-1')
client = chronos2_client.Chronos2Client(endpoint_name='chronos2-demo-ep', runtime=runtime)

chr_paths = chronos2_client.simulate_chronos2(
    client, panel, as_of=AS_OF, horizon=HORIZON, n_paths=N_PATHS,
    covariates=covs,
)
print(chr_paths.source, chr_paths.paths.shape)

## Beat 5 — Blend by w_stress

In [ ]:
par_terminal = exposure.portfolio_mtm(swap_book, par_paths)[:, -1]
chr_terminal = exposure.portfolio_mtm(swap_book, chr_paths)[:, -1]

blended = blender.blend_paths(
    par_paths, chr_paths, par_terminal, chr_terminal,
    w_stress=snap.w_stress,
)
print(blended.source, blended.paths.shape, 'weights sum =', blended.weights.sum())

## Beat 6 — Compute exposure metrics

In [ ]:
baseline_metrics = exposure.compute_metrics(swap_book, par_paths)
augmented_metrics = exposure.compute_metrics(swap_book, blended)

print('PFE@97.5 terminal:')
print(f'  baseline:  {baseline_metrics["pfe_975"][-1]:,.0f} USD')
print(f'  augmented: {augmented_metrics["pfe_975"][-1]:,.0f} USD')
print(f'  delta:     {augmented_metrics["pfe_975"][-1] - baseline_metrics["pfe_975"][-1]:,.0f} USD '
      f'({100 * (augmented_metrics["pfe_975"][-1] / baseline_metrics["pfe_975"][-1] - 1):.1f}%)')

## Beat 7 — Money-shot overlay + diagnostics

In [ ]:
fig = pfe_overlay.plot_pfe_overlay(
    baseline=baseline_metrics, augmented=augmented_metrics,
    title=f'XCCY book PFE@97.5 — as-of {AS_OF} (w_stress={snap.w_stress:.2f})',
)
fig


In [ ]:
w_stress_full = (
    pd.Series(0.0, index=panel.index)
      .combine_first(__import__('chronos2_demo.data.regimes', fromlist=['w_stress_series'])
                     .w_stress_series(panel, window=20))
      .reindex(panel.index)
      .ffill().fillna(0.0)
)
fig = regime_ribbon.plot_regime_ribbon(
    w_stress_full.tail(252), threshold=0.7, title='w_stress trailing 252 BD',
)
fig

In [ ]:
fig = path_samples.plot_path_fan(
    par_paths, series='USDEUR', n_path_samples=40,
    title='Parametric paths — USDEUR',
)
fig


In [ ]:
fig = path_samples.plot_path_fan(
    chr_paths, series='USDEUR', n_path_samples=40,
    title='Chronos-2 paths — USDEUR',
)
fig


## Diagnostics — does the augmented set match reality better?

In [ ]:
exc_par = tail_dependence.exceedance_correlation(par_paths, 'USDEUR', 'USD_5Y', quantile=0.05)
exc_chr = tail_dependence.exceedance_correlation(chr_paths, 'USDEUR', 'USD_5Y', quantile=0.05)
exc_realized = tail_dependence.realized_exceedance_correlation(panel, 'USDEUR', 'USD_5Y', quantile=0.05)
print(f'exceedance corr USDEUR x USD_5Y:')
print(f'  parametric: {exc_par:+.3f}')
print(f'  chronos-2:  {exc_chr:+.3f}')
print(f'  realized:   {exc_realized:+.3f}')